# Figure 5 Microglia Workflow

Purpose: clean, publication-facing code for Figure 5. The notebook keeps only the final analytical logic: microglial subclustering, marker confirmation, functional scoring, spatial enrichment, astrocyte-microglia communication, and immunostaining validation.

All paths are virtual placeholders. The code is organized for readability and traceability, not for immediate execution.


In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(ggpubr)
  library(scales)
  library(patchwork)
  library(viridis)
  library(pheatmap)
  library(readxl)
  library(svglite)
  library(ragg)
  library(grid)
})

# Virtual project paths: replace these with real paths only when reproducing the figures.
PROJECT_DIR <- "/path/to/hippocampal_sclerosis_project"
DATA_DIR    <- file.path(PROJECT_DIR, "data")
RESULT_DIR  <- file.path(PROJECT_DIR, "results")
FIG_DIR     <- file.path(PROJECT_DIR, "figure_exports")
SRC_DIR     <- file.path(PROJECT_DIR, "source_data")

dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(SRC_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_LEVELS <- c("HS-", "HS+")
GROUP_COLORS <- c("HS-" = "#8DC7C2", "HS+" = "#E94C5F")
DIVERGING_COLORS <- c("#14448C", "#5076C1", "#92A6DE", "#CED7F2",
                      "#F6F6F6", "#F5CECE", "#DE9494", "#B85A5B", "#7D2828")
EXPRESSION_COLORS <- c("#440154", "#31688E", "#35B779", "#FDE725")

theme_pub <- function(base_size = 7) {
  theme_classic(base_size = base_size, base_family = "Arial") +
    theme(
      axis.line = element_line(linewidth = 0.25, colour = "black"),
      axis.ticks = element_line(linewidth = 0.25, colour = "black"),
      axis.text = element_text(colour = "black"),
      legend.key.height = unit(3.5, "mm"),
      legend.key.width = unit(3.5, "mm"),
      legend.title = element_text(size = base_size),
      legend.text = element_text(size = base_size - 1),
      strip.background = element_rect(fill = "grey92", colour = "black", linewidth = 0.25),
      strip.text = element_text(colour = "black", face = "plain")
    )
}
theme_set(theme_pub())

save_pub <- function(plot, file_stub, width_mm, height_mm, dpi = 600) {
  svg_file <- file.path(FIG_DIR, paste0(file_stub, ".svg"))
  pdf_file <- file.path(FIG_DIR, paste0(file_stub, ".pdf"))
  tif_file <- file.path(FIG_DIR, paste0(file_stub, ".tiff"))

  svglite::svglite(svg_file, width = width_mm / 25.4, height = height_mm / 25.4)
  print(plot)
  dev.off()

  cairo_pdf(pdf_file, width = width_mm / 25.4, height = height_mm / 25.4, family = "Arial")
  print(plot)
  dev.off()

  ragg::agg_tiff(tif_file, width = width_mm, height = height_mm, units = "mm",
                 res = dpi, compression = "lzw")
  print(plot)
  dev.off()

  invisible(c(svg = svg_file, pdf = pdf_file, tiff = tif_file))
}

clean_group <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    x %in% c("HS-", "Normal", "Control", "CTL", "TLE-noHS") ~ "HS-",
    x %in% c("HS+", "HS", "Sclerosis", "TLE-HS") ~ "HS+",
    TRUE ~ x
  )
}

sig_label <- function(p) {
  dplyr::case_when(
    is.na(p) ~ "ns",
    p < 0.0001 ~ "****",
    p < 0.001 ~ "***",
    p < 0.01 ~ "**",
    p < 0.05 ~ "*",
    TRUE ~ "ns"
  )
}

first_existing_col <- function(object, candidates) {
  hit <- candidates[candidates %in% colnames(object@meta.data)]
  if (length(hit) == 0) {
    stop("None of these metadata columns were found: ", paste(candidates, collapse = ", "))
  }
  hit[[1]]
}

plot_score_umap <- function(object, score_col, title, highlight_cluster = NULL,
                            file_stub = NULL, width_mm = 48, height_mm = 42) {
  score_col <- first_existing_col(object, score_col)
  emb <- as.data.frame(Embeddings(object, "umap"))
  colnames(emb)[1:2] <- c("UMAP_1", "UMAP_2")
  emb$score <- object@meta.data[[score_col]]
  emb$subcluster <- object$subcluster

  p <- ggplot(emb, aes(UMAP_1, UMAP_2, colour = score)) +
    geom_point(size = 0.08, stroke = 0, alpha = 0.85) +
    scale_colour_gradientn(colours = EXPRESSION_COLORS, name = "Expression") +
    labs(title = title, x = "UMAP1", y = "UMAP2") +
    coord_equal() +
    theme_pub() +
    theme(
      legend.position = c(0.78, 0.12),
      legend.background = element_blank(),
      axis.text = element_blank(),
      axis.ticks = element_blank()
    )

  if (!is.null(highlight_cluster)) {
    centroid <- emb |>
      filter(subcluster == highlight_cluster) |>
      summarise(x = median(UMAP_1), y = median(UMAP_2))
    p <- p + annotate("text", x = centroid$x, y = centroid$y, label = title,
                      colour = "white", size = 2.2, fontface = "bold")
  }

  if (!is.null(file_stub)) save_pub(p, file_stub, width_mm, height_mm)
  p
}

plot_group_violin <- function(object, cluster, score_col, ylab, file_stub,
                              width_mm = 32, height_mm = 42) {
  score_col <- first_existing_col(object, score_col)
  dat <- object@meta.data |>
    transmute(subcluster, Group, score = .data[[score_col]]) |>
    filter(subcluster == cluster, Group %in% GROUP_LEVELS) |>
    mutate(Group = factor(Group, levels = GROUP_LEVELS))

  pval <- tryCatch(wilcox.test(score ~ Group, data = dat)$p.value, error = function(e) NA_real_)

  p <- ggplot(dat, aes(Group, score, fill = Group)) +
    geom_violin(width = 0.9, linewidth = 0.15, colour = NA, trim = TRUE) +
    geom_boxplot(width = 0.18, linewidth = 0.25, outlier.shape = NA,
                 fill = "white", colour = "grey30") +
    annotate("text", x = 1.5, y = max(dat$score, na.rm = TRUE) * 1.05,
             label = sig_label(pval), size = 2.5) +
    scale_fill_manual(values = GROUP_COLORS) +
    labs(x = NULL, y = ylab, title = paste0(cluster, " in disease axis")) +
    theme_pub() +
    theme(legend.position = "none")

  write.csv(dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}

make_density_bar <- function(dat, marker_label, file_stub, width_mm = 34, height_mm = 38) {
  plot_dat <- dat |>
    mutate(Group = factor(clean_group(Group), levels = GROUP_LEVELS)) |>
    group_by(Group) |>
    summarise(mean = mean(density_mm2, na.rm = TRUE),
              sem = sd(density_mm2, na.rm = TRUE) / sqrt(dplyr::n()),
              .groups = "drop")
  pval <- tryCatch(wilcox.test(density_mm2 ~ Group, data = dat)$p.value, error = function(e) NA_real_)

  p <- ggplot(plot_dat, aes(Group, mean, fill = Group)) +
    geom_col(width = 0.58, colour = "black", linewidth = 0.25) +
    geom_errorbar(aes(ymin = mean - sem, ymax = mean + sem), width = 0.16, linewidth = 0.25) +
    annotate("text", x = 1.5, y = max(plot_dat$mean + plot_dat$sem, na.rm = TRUE) * 1.12,
             label = sig_label(pval), size = 2.6) +
    scale_fill_manual(values = GROUP_COLORS) +
    labs(x = NULL, y = expression("Cell density (/mm"^2*")"), title = marker_label) +
    theme_pub() +
    theme(legend.position = "none")

  write.csv(dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}


In [ ]:
MICRO_COLORS <- c(
  "Micro.0" = "#E6EE9C",
  "Micro.1" = "#D4E157",
  "Micro.2" = "#9E9D24",
  "Micro.3" = "#F4D70B",
  "Micro.4" = "#AFC127"
)

micro <- readRDS(file.path(DATA_DIR, "seurat", "Micro_subcluster.rds"))
DefaultAssay(micro) <- "RNA"
micro <- NormalizeData(micro, verbose = FALSE)

# Use curated labels when present. The fallback is only a transparent placeholder
# for public code release; do not merge Micro.1 and Micro.4.
if (!"subcluster" %in% colnames(micro@meta.data)) {
  cluster_col <- "SCT_snn_res.0.8"
  stopifnot(cluster_col %in% colnames(micro@meta.data))
  micro$subcluster <- dplyr::recode(
    as.character(micro@meta.data[[cluster_col]]),
    "0" = "Micro.0",
    "1" = "Micro.1",
    "2" = "Micro.2",
    "3" = "Micro.3",
    "4" = "Micro.4",
    .default = NA_character_
  )
}

micro$subcluster <- factor(micro$subcluster, levels = names(MICRO_COLORS))
micro$Group <- factor(clean_group(micro$Group), levels = GROUP_LEVELS)
Idents(micro) <- "subcluster"

write.csv(micro@meta.data, file.path(SRC_DIR, "fig5_micro_metadata.csv"))
table(micro$subcluster, micro$Group)


In [ ]:
# Figure 5a: microglial subtype UMAP.
p5a <- DimPlot(
  micro, reduction = "umap", group.by = "subcluster",
  cols = MICRO_COLORS, pt.size = 0.08, label = TRUE, repel = TRUE
) +
  labs(title = paste0("n = ", format(ncol(micro), big.mark = ",")),
       x = "UMAP1", y = "UMAP2") +
  coord_equal() +
  theme_pub() +
  theme(axis.text = element_blank(), axis.ticks = element_blank())
save_pub(p5a, "Fig5a_Micro_subcluster_UMAP", 62, 52)
p5a

# Figure 5b: marker confirmation. scale = FALSE keeps original average expression.
micro_marker_genes <- rev(c(
  "FRMD4A", "SFMBT2", "TUBB2A", "IL1RAPL1", "TF",
  "APOE", "A2M", "CD74", "CCL2", "IL1B"
))
p5b <- DotPlot(micro, features = micro_marker_genes, group.by = "subcluster", scale = FALSE) +
  coord_flip() +
  scale_colour_gradientn(colours = DIVERGING_COLORS, name = "Expression") +
  scale_size(range = c(0.2, 4.2), name = "Ratio") +
  labs(x = NULL, y = NULL) +
  theme_pub() +
  theme(axis.text.x = element_text(angle = 60, hjust = 1),
        panel.border = element_rect(fill = NA, colour = "black", linewidth = 0.25))
save_pub(p5b, "Fig5b_Micro_marker_dotplot", 50, 66)
p5b


In [ ]:
# Optional reproducible marker discovery and GO enrichment.
run_marker_enrichment <- FALSE

if (run_marker_enrichment) {
  suppressPackageStartupMessages({
    library(clusterProfiler)
    library(org.Hs.eg.db)
  })

  micro_markers_all <- FindAllMarkers(
    micro,
    only.pos = TRUE,
    test.use = "wilcox",
    min.pct = 0.10,
    logfc.threshold = 0.25
  )

  micro_marker_leaders <- micro_markers_all |>
    filter(p_val_adj < 0.05) |>
    group_by(cluster) |>
    slice_max(avg_log2FC, n = 200, with_ties = FALSE) |>
    ungroup()

  micro_go <- micro_marker_leaders |>
    group_by(cluster) |>
    group_modify(~ {
      entrez <- bitr(.x$gene, fromType = "SYMBOL", toType = "ENTREZID",
                     OrgDb = org.Hs.eg.db) |> pull(ENTREZID) |> unique()
      enrichGO(
        gene = entrez, OrgDb = org.Hs.eg.db, ont = "BP",
        pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.20,
        readable = TRUE
      ) |> as.data.frame()
    }) |>
    ungroup()

  write.csv(micro_markers_all, file.path(SRC_DIR, "fig5_micro_FindAllMarkers.csv"), row.names = FALSE)
  write.csv(micro_go, file.path(SRC_DIR, "fig5_micro_GO_BP_all.csv"), row.names = FALSE)
}


In [ ]:
# Figure 5c: curated microglial BP modules.
# Expected columns: Description, geneID. geneID uses "/"-separated HGNC symbols.
micro_bp <- readxl::read_xlsx(file.path(DATA_DIR, "curated_gene_sets", "Micro_BP_curated.xlsx")) |>
  transmute(
    module = Description,
    genes = str_split(geneID, "/", simplify = FALSE)
  )

for (i in seq_len(nrow(micro_bp))) {
  micro <- AddModuleScore(
    object = micro,
    features = list(micro_bp$genes[[i]]),
    name = micro_bp$module[[i]],
    assay = "RNA"
  )
}

micro_bp_cols <- paste0(micro_bp$module, "1")
micro_bp_matrix <- micro@meta.data |>
  select(subcluster, all_of(micro_bp_cols)) |>
  group_by(subcluster) |>
  summarise(across(all_of(micro_bp_cols), mean, na.rm = TRUE), .groups = "drop") |>
  tibble::column_to_rownames("subcluster") |>
  as.matrix()

micro_bp_z <- t(scale(t(micro_bp_matrix)))
micro_bp_z[is.na(micro_bp_z)] <- 0

pheatmap(
  micro_bp_z,
  color = colorRampPalette(c("#F6F5EB", "#FEE0D2", "#DE2D26", "#7F1818"))(100),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = "grey75",
  fontsize = 7,
  filename = file.path(FIG_DIR, "Fig5c_Micro_BP_heatmap.pdf"),
  width = 3.4,
  height = 3.0
)
write.csv(micro_bp_z, file.path(SRC_DIR, "fig5c_micro_BP_heatmap_zscore.csv"))


In [ ]:
# Figure 5d-k: functional microglial axes.
micro_function_panels <- tibble::tribble(
  ~panel, ~cluster,  ~title,                                  ~score_candidates,
  "d/e",  "Micro.0", "GTPase regulator activity",             list(c("GTPase regulator activity1", "GTPase.regulator.activity1")),
  "f/g",  "Micro.1", "R. of trans-synaptic signaling",        list(c("regulation of trans synaptic signaling1", "regulation.of.trans.synaptic.signaling1", "regulation.of.synaptic.signaling1")),
  "h/i",  "Micro.3", "DAM",                                   list(c("DAM1", "Disease associated microglia1")),
  "j/k",  "Micro.4", "R. of inflammatory response",           list(c("Regulation of inflammatory response1", "Regulation.of.inflammatory.response1", "response.to.inflammatory.response1"))
)

p5d <- plot_score_umap(micro, micro_function_panels$score_candidates[[1]],
                       "GTPase regulator activity", "Micro.0", "Fig5d_Micro0_GTPase_UMAP")
p5e <- plot_group_violin(micro, "Micro.0", micro_function_panels$score_candidates[[1]],
                         "GTPase regulator activity", "Fig5e_Micro0_GTPase_violin")

p5f <- plot_score_umap(micro, micro_function_panels$score_candidates[[2]],
                       "R. of trans-synaptic signaling", "Micro.1", "Fig5f_Micro1_synaptic_UMAP")
p5g <- plot_group_violin(micro, "Micro.1", micro_function_panels$score_candidates[[2]],
                         "R. of trans-synaptic signaling", "Fig5g_Micro1_synaptic_violin")

p5h <- plot_score_umap(micro, micro_function_panels$score_candidates[[3]],
                       "DAM", "Micro.3", "Fig5h_Micro3_DAM_UMAP")
p5i <- plot_group_violin(micro, "Micro.3", micro_function_panels$score_candidates[[3]],
                         "DAM", "Fig5i_Micro3_DAM_violin")

p5j <- plot_score_umap(micro, micro_function_panels$score_candidates[[4]],
                       "R. of inflammatory response", "Micro.4", "Fig5j_Micro4_inflammatory_UMAP")
p5k <- plot_group_violin(micro, "Micro.4", micro_function_panels$score_candidates[[4]],
                         "R. of inflammatory response", "Fig5k_Micro4_inflammatory_violin")


In [ ]:
# Figure 5m/n: Micro.4 spatial enrichment in FAS.
spatial_meta <- readRDS(file.path(DATA_DIR, "spatial", "cellbin_spatial_meta.rds"))
spatial_meta <- spatial_meta |>
  mutate(Group = factor(clean_group(Group), levels = GROUP_LEVELS),
         subcluster = factor(subcluster, levels = names(MICRO_COLORS)))

plot_spatial_state <- function(dat, state, region, file_stub, width_mm = 58, height_mm = 36) {
  plot_dat <- dat |>
    filter(region == !!region, subcluster %in% names(MICRO_COLORS)) |>
    mutate(is_state = subcluster == state)

  p <- ggplot(plot_dat, aes(x, y)) +
    geom_point(data = filter(plot_dat, !is_state), colour = "grey82", size = 0.05, alpha = 0.35) +
    geom_point(data = filter(plot_dat, is_state), aes(colour = subcluster), size = 0.18, alpha = 0.9) +
    scale_colour_manual(values = MICRO_COLORS, drop = FALSE) +
    facet_grid(. ~ Group) +
    coord_equal() +
    labs(title = paste0(state, " in ", region), x = NULL, y = NULL) +
    theme_pub() +
    theme(axis.text = element_blank(), axis.ticks = element_blank(), legend.position = "none")

  write.csv(plot_dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}

p5m <- plot_spatial_state(spatial_meta, "Micro.4", "FAS", "Fig5m_Micro4_FAS_spatial")

micro4_density <- spatial_meta |>
  filter(region == "FAS", subcluster == "Micro.4") |>
  count(sample, Group, name = "positive_count") |>
  left_join(
    spatial_meta |> filter(region == "FAS") |> distinct(sample, Group, area_mm2),
    by = c("sample", "Group")
  ) |>
  mutate(density_mm2 = positive_count / area_mm2)

p5n <- make_density_bar(micro4_density, "Micro.4", "Fig5n_Micro4_FAS_density")


In [ ]:
# Figure 5p/r/s: spatial cell-cell communication summaries from StereoSiTE-like output.

AREAS <- c("Alveus", "SO", "SL", "SR", "SM")

collect_stereosite <- function(result_dir, chips = CHIPS, areas = AREAS, p_cutoff = 0.05) {
  files <- expand.grid(chip = chips, area = areas, stringsAsFactors = FALSE) |>
    mutate(file = file.path(result_dir, "area_m", paste0(area, "_", chip, "_merge.csv")))

  purrr::pmap_dfr(files, function(chip, area, file) {
    if (!file.exists(file)) return(NULL)
    read.csv(file) |>
      mutate(sample = chip, area_m = area) |>
      filter(pval < p_cutoff, value > 0)
  })
}

plot_distance_profile <- function(dat, state, target, file_stub, width_mm = 42, height_mm = 46) {
  plot_dat <- dat |>
    filter(cell_state == state, target_compartment %in% c("Alveus", "SO", "SL", "SR", "SM")) |>
    group_by(Group, target_compartment) |>
    summarise(mean_density = mean(density, na.rm = TRUE),
              sem = sd(density, na.rm = TRUE) / sqrt(dplyr::n()),
              .groups = "drop") |>
    mutate(target_compartment = factor(target_compartment, levels = c("Alveus", "SO", "SL", "SR", "SM")))

  p <- ggplot(plot_dat, aes(target_compartment, mean_density, group = Group, colour = Group)) +
    geom_line(linewidth = 0.35) +
    geom_point(size = 1.0) +
    geom_errorbar(aes(ymin = mean_density - sem, ymax = mean_density + sem),
                  width = 0.12, linewidth = 0.25) +
    scale_colour_manual(values = GROUP_COLORS) +
    labs(x = NULL, y = target, title = state) +
    theme_pub()

  write.csv(plot_dat, file.path(SRC_DIR, paste0(file_stub, "_source.csv")), row.names = FALSE)
  save_pub(p, file_stub, width_mm, height_mm)
  p
}

stereosite_result <- collect_stereosite(file.path(RESULT_DIR, "Stereosite"))

# Figure 5p: laminar density profile of the homeostatic-to-reactive axis.
# Expected columns: sample, Group, cell_state, target_compartment, density.
laminar_density <- read.csv(file.path(DATA_DIR, "spatial", "micro_astro_laminar_density.csv")) |>
  mutate(Group = factor(clean_group(Group), levels = GROUP_LEVELS))

p5p_micro <- plot_distance_profile(
  laminar_density,
  state = "H.micro",
  target = "Density in R.micro-I",
  file_stub = "Fig5p_Hmicro_laminar_density"
)

p5p_astro <- plot_distance_profile(
  laminar_density,
  state = "R.micro",
  target = "Density in A.astro",
  file_stub = "Fig5p_Rmicro_Aastro_laminar_density"
)

interaction_summary <- stereosite_result |>
  mutate(Group = factor(clean_group(Group), levels = GROUP_LEVELS))

if ("interaction_name_2" %in% colnames(interaction_summary)) {
  interaction_summary <- interaction_summary |>
    mutate(interaction = interaction_name_2)
} else {
  interaction_summary <- interaction_summary |>
    mutate(interaction = paste0(ligand, " -> ", receptor))
}

interaction_summary <- interaction_summary |>
  group_by(Group, area_m, celltype1, celltype2, interaction) |>
  summarise(relative_strength = mean(value, na.rm = TRUE), .groups = "drop")

selected_interactions <- c(
  "CX3CL1 -> CX3CR1",
  "CCL2 -> CCR2",
  "TGFB1 -> TGFBR2",
  "ITGAM -> ITGB2"
)

p5r <- interaction_summary |>
  filter(interaction %in% selected_interactions,
         celltype1 %in% c("A.astro", "R.micro-III"),
         celltype2 %in% c("R.micro-II", "A.astro")) |>
  ggplot(aes(interaction, paste(celltype1, celltype2, sep = "|"), fill = relative_strength)) +
  geom_tile(colour = "grey80", linewidth = 0.25) +
  facet_grid(area_m ~ Group, scales = "free_y", space = "free_y") +
  scale_fill_gradientn(colours = c("#F7FBFF", "#9ECAE1", "#6A51A3"),
                       name = "Relative strength") +
  labs(x = NULL, y = NULL) +
  theme_pub() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5))
save_pub(p5r, "Fig5r_Astro_Micro_interaction_heatmap", 105, 48)

p5s <- interaction_summary |>
  filter(interaction == "CX3CL1 -> CX3CR1") |>
  ggplot(aes(x = celltype1, y = celltype2, size = relative_strength, colour = Group)) +
  geom_point(alpha = 0.9) +
  facet_grid(. ~ Group) +
  scale_colour_manual(values = GROUP_COLORS) +
  scale_size(range = c(0.5, 4.5), name = "Interaction intensity") +
  labs(title = "CX3CL1 -> CX3CR1", x = "Sender", y = "Receiver") +
  theme_pub() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
save_pub(p5s, "Fig5s_CX3CL1_CX3CR1_dotmap", 68, 40)


In [ ]:
# Figure 5l/t: immunostaining validation statistics.
# Expected input table contains one row per field of view:
# sample, Group, region, marker_set, positive_count, area_mm2.
micro_validation <- read.csv(file.path(DATA_DIR, "validation", "micro_immunostaining_density.csv")) |>
  mutate(
    Group = factor(clean_group(Group), levels = GROUP_LEVELS),
    density_mm2 = positive_count / area_mm2
  )

validation_panels <- tibble::tribble(
  ~marker_set,          ~file_stub,
  "IBA1+CCL2",          "Fig5l_IBA1_CCL2_density",
  "S100B+IBA1+CCL2",    "Fig5t_S100B_IBA1_CCL2_density"
)

micro_density_plots <- purrr::pmap(
  validation_panels,
  function(marker_set, file_stub) {
    micro_validation |>
      filter(marker_set == !!marker_set) |>
      make_density_bar(marker_set, file_stub)
  }
)
